<a href="https://colab.research.google.com/github/Jasonmiller513/Dissertation/blob/Updates/STEP_5_STATE_SPACE_DISCRETIZATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STEP 5: STATE-SPACE DISCRETIZATION

# Feature Selection

In [ ]:
state_features = [
    'Profit Margin',
    'Shipping Cost Ratio',
    'Historical On-Time Rate',
    'Regional Delay Score',
    'Product Category Id'
]

# Fit Bins Using Training Data Only

In [ ]:
# Load scaled training data
train_df = pd.read_csv("dataco_rl_train_scaled.csv")

exclude_cols = [
    'Reward',
    'Late_delivery_risk',
    'Action_ID'
]

continuous_features = [
    col for col in train_df.select_dtypes(include='number').columns
    if col not in exclude_cols
]

bin_edges = {}

# Create 10 bins for each feature
for col in continuous_features:

    _, bins = pd.qcut(
        train_df[col],
        q=10,
        labels=False,
        retbins=True,
        duplicates='drop'
    )

    bin_edges[col] = bins

# Save bin definitions
joblib.dump(
    bin_edges,
    "state_bins.pkl"
)

print("Bins created successfully")

In [ ]:
bin_counts = {}

for col in continuous_features:
    _, bins = pd.qcut(
        train_df[col],
        q=10,
        retbins=True,
        duplicates='drop'
    )

    bin_counts[col] = len(bins) - 1

print(bin_counts)

# Apply Bins to Train/Test Data

In [ ]:
train_df = pd.read_csv("dataco_rl_train_scaled.csv")
test_df = pd.read_csv("dataco_rl_test_scaled.csv")

bin_edges = joblib.load("state_bins.pkl")

continuous_features = list(bin_edges.keys())

for col in continuous_features:

    train_df[col + "_bin"] = np.digitize(
        train_df[col],
        bin_edges[col][1:-1]
    )

    test_df[col + "_bin"] = np.digitize(
        test_df[col],
        bin_edges[col][1:-1]
    )

train_df.to_csv(
    "dataco_rl_train_discrete.csv",
    index=False
)

test_df.to_csv(
    "dataco_rl_test_discrete.csv",
    index=False
)

print("Discretization complete")

# Build RL States

In [ ]:
# Redefine state_columns with existing features in train_df
state_columns = [
    'Product Category Id',
    'Category Name',
    'shipping_mode_speed',
    'stocking_score_bin',
    'margin_pct_bin',
    'shipping_delay_bin',
    'profit_per_item_bin',
    'unique_products_bin'
]

train_df['state'] = train_df[state_columns] \
    .astype(str) \
    .agg('|'.join, axis=1)

# Create State IDs

In [ ]:
state_encoder = LabelEncoder()

train_df['state_id'] = state_encoder.fit_transform(
    train_df['state']
)

joblib.dump(
    state_encoder,
    "state_encoder.pkl"
)

print(
    "Unique states:",
    train_df['state_id'].nunique()
)

# Create Action IDs

In [ ]:
candidate_regions = pd.read_csv(
    "candidate_fulfillment_regions.csv"
)

shipping_modes = [
    "Standard Class",
    "Second Class",
    "First Class",
    "Same Day"
]

actions = []

for region in candidate_regions["Order Region"].unique():
    for mode in shipping_modes:
        actions.append(f"{region}|{mode}")

from sklearn.preprocessing import LabelEncoder

action_encoder = LabelEncoder()

action_ids = action_encoder.fit_transform(actions)

action_mapping = pd.DataFrame({
    "action": actions,
    "action_id": action_ids
})

action_mapping.to_csv(
    "action_mapping.csv",
    index=False
)

In [ ]:
action = (
    candidate_regions,
    shipping_modes
)

In [ ]:
action_encoder = LabelEncoder()

# Reconstruct 'Shipping Mode' string from 'shipping_mode_speed'
shipping_mode_map_reverse = {
    1: 'Standard Class',
    2: 'Second Class',
    3: 'First Class',
    4: 'Same Day'
}
train_df['Shipping Mode_str'] = train_df['shipping_mode_speed'].map(shipping_mode_map_reverse)

# Reconstruct 'Order Region' string from one-hot encoded columns
order_region_cols = [col for col in train_df.columns if col.startswith('Order Region_')]
# Use idxmax to find the column with value 1, then strip the prefix
train_df['Order Region_str'] = train_df[order_region_cols].idxmax(axis=1).str.replace('Order Region_', '')

# Create the 'action' column by combining the reconstructed strings
train_df['action'] = train_df['Order Region_str'] + '|' + train_df['Shipping Mode_str']

# Now fit and transform the action column
train_df['action_id'] = action_encoder.fit_transform(
    train_df['action']
)

joblib.dump(
    action_encoder,
    "action_encoder.pkl"
)

print(
    "Unique actions:",
    train_df['action_id'].nunique()
)

# Check State Space Size

In [ ]:
print(
    "Unique states:",
    train_df['state'].nunique()
)

print(
    "Unique actions:",
    train_df['action'].nunique()
)

print(
    "Q-table size:",
    train_df['state'].nunique()
    *
    train_df['action'].nunique()
)